In [1]:
from transformers import pipeline

In [2]:
#!pip install timm

In [3]:
# Specify the inference task
object_detection = pipeline(
    task="object-detection",
    model="facebook/detr-resnet-50",
)

Loading weights:   0%|          | 0/530 [00:00<?, ?it/s]

DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


preprocessor_config.json:   0%|          | 0.00/290 [00:00<?, ?B/s]

The image processor of type `DetrImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [5]:
from PIL import Image

In [7]:
traffic_img = Image.open("images/image_2.jpeg")

In [8]:
detections = object_detection(traffic_img)

detections[:4]

[{'score': 0.6827951073646545,
  'label': 'car',
  'box': {'xmin': 1904, 'ymin': 922, 'xmax': 2016, 'ymax': 1025}},
 {'score': 0.9888834357261658,
  'label': 'car',
  'box': {'xmin': 346, 'ymin': 1726, 'xmax': 1081, 'ymax': 2286}},
 {'score': 0.5736234188079834,
  'label': 'person',
  'box': {'xmin': 1843, 'ymin': 831, 'xmax': 1948, 'ymax': 942}},
 {'score': 0.8973751068115234,
  'label': 'car',
  'box': {'xmin': 1807, 'ymin': 1124, 'xmax': 1937, 'ymax': 1291}}]

In [9]:
from collections import Counter

# Extract labels
labels = [d['label'] for d in detections]

# Count occurrences
object_counts = Counter(labels)

print(object_counts)

Counter({'car': 29, 'person': 21, 'truck': 5, 'bicycle': 2, 'bus': 1})


In [10]:
def counts_to_text(object_counts):
    if not object_counts:
        return "No objects were detected in the image."
    
    phrases = []
    
    for obj, count in object_counts.items():
        if count == 1:
            phrases.append(f"1 {obj}")
        else:
            phrases.append(f"{count} {obj}s")
    
    # Join properly with commas and 'and'
    if len(phrases) == 1:
        return f"The image contains {phrases[0]}."
    else:
        return "The image contains " + ", ".join(phrases[:-1]) + \
               " and " + phrases[-1] + "."

In [11]:
description = counts_to_text(object_counts)
print(description)

The image contains 29 cars, 21 persons, 2 bicycles, 5 trucks and 1 bus.


In [15]:
narrator = pipeline(
    task="text-to-speech",
    model="suno/bark-small",
)

Loading weights:   0%|          | 0/542 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_layers.1.weight to fine_acoustics.lm_heads.0.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_layers.2.weight to fine_acoustics.lm_heads.1.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_layers.3.weight to fine_acoustics.lm_heads.2.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie fine_acoustics.input_embeds_l

In [13]:
text = description
text

'The image contains 29 cars, 21 persons, 2 bicycles, 5 trucks and 1 bus.'

In [16]:
narrated_text = narrator(text)

Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate', 'min_eos_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [17]:
print(narrated_text["sampling_rate"])

24000


In [18]:
narrated_text["audio"][0]

np.float32(0.00082413637)

In [19]:
from IPython.display import Audio as IPythonAudio

IPythonAudio(
    narrated_text["audio"],
    rate=narrated_text["sampling_rate"]
)